# Fraud Classification Training

Notebook pipeline for reading one training dataset that contains both features and the target label, cleaning/filtering rows, preparing features, and training three classification models:

- Logistic Regression
- Random Forest
- XGBoost

Expected input: one CSV file with transaction/app-event fields plus one target column such as `label`, `is_fraud`, `target`, or `fraud_label`.

Required packages if your Jupyter environment does not already have them:

```bash
pip install pandas numpy scikit-learn xgboost joblib
```

In [52]:
#!pip install pandas numpy scikit-learn xgboost joblib


In [53]:
from pathlib import Path
import platform
import struct
import sys

import numpy as np
import pandas as pd

if struct.calcsize("P") * 8 != 64:
    raise RuntimeError(
        f"This notebook needs 64-bit Python for numpy/scipy/scikit-learn. "
        f"Current kernel: {sys.executable} ({platform.platform()}). "
        "Create/select a 64-bit Python kernel and reinstall requirements."
    )

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_curve,
        roc_auc_score,
    )
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
except (ImportError, KeyError) as exc:
    raise RuntimeError(
        "scikit-learn failed to import in this notebook kernel. "
        "This usually means the kernel is 32-bit or sklearn/scipy was installed "
        "for the wrong architecture. Recreate the environment with 64-bit Python, "
        "then run: python -m pip install --force-reinstall -r requirements.txt"
    ) from exc

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

pd.set_option("display.max_columns", 120)
RANDOM_STATE = 42


## 1. Read Dataset

In [54]:
DATA_PATH = Path("datasets/fraud_events_100k.csv")

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(100000, 27)


/tmp/ipykernel_376/308409114.py:3: DtypeWarning: Columns (14,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,tx_id,sender_id,recipient_id,amount,currency,device_id,device_type,device_trusted,sender_ip,sender_lat,sender_lon,sender_city,timestamp,is_fraud,fraud_type,sender_account_age_days,sender_monthly_tx_count,sender_avg_amount,label,event_id,event_timestamp,pin_failures,device_changed,new_device_id,is_offhours_login,session_duration_sec,app_version
0,7b01c9a5-b4f6-4721-b6c5-4ed7e07fd8d0,7958ef20-8915-42d6-b035-813972eebf75,9db5768c-b415-4468-9dda-15ce07986d70,1407.21,GBP,97843f45-22fc-46cb-b461-33c6c2c08349,web,True,208.144.172.203,52.468246,16.854304,Gdansk,2026-01-01T00:00:00+00:00,True,geo_anomaly,4,49,4461.59,1,c3a572c7-c42e-4dba-bc9b-7b051d8e7b5a,2026-01-01T00:00:00+00:00,0,False,NaN,False,62,3.3.4
1,dbd08b62-ba6f-4c2a-b63d-416cd8b79942,8a206ca3-7656-48a1-96a9-848b5a60495a,bb6c93e1-e3aa-419c-9b34-3a9bef328b61,79.79,GBP,3c10f107-a98e-41c9-916c-db12f80c02e7,android,True,216.114.74.97,51.170816,17.073162,Wroclaw,2026-01-01T00:00:02+00:00,False,NaN,2643,36,100.73,0,6eefb0fa-9d0a-4854-affa-697e2fec2496,2026-01-01T00:00:02+00:00,0,False,NaN,False,21,3.3.8
2,2dd29671-6bf8-46c6-b57b-bb6024d5af22,61a7733e-b3c8-4018-ad06-3acbeed3c5b2,d68b7552-dbe8-4138-91a7-a046177f677b,2438.57,USD,6ef7925e-fd4b-4e1d-a999-029ea66e2119,ios,True,33.190.163.17,52.143281,21.117524,Warsaw,2026-01-01T00:00:05+00:00,False,NaN,29,21,2361.50,0,cf4e0dc1-a709-4f5f-8255-03759f295f37,2026-01-01T00:00:05+00:00,0,False,NaN,False,293,3.9.9
3,d835f341-012d-4339-91cf-8181c1d68637,5d9447e9-9e2b-4870-84a6-498582945a80,72d59ee0-5477-4e0b-b4b2-5aa957de3a9a,93.99,USD,9eb48c8b-0b43-4ced-9ca3-1fb00bd540a6,web,True,105.148.255.196,54.327010,18.736589,Gdansk,2026-01-01T00:00:06+00:00,False,NaN,1638,44,106.09,0,5e50674c-31b9-40b3-ba79-f6e052fa18af,2026-01-01T00:00:06+00:00,0,False,NaN,False,248,3.0.3
4,0d971971-77f4-4ffe-90ac-b013d4020dbf,5f89faaa-46f4-43ad-96c5-14b494f7cd82,100146c2-c267-4839-8f83-065fba328e4d,207.06,USD,541ccc3b-bef5-4a08-8fc4-e8114ae97cab,web,True,219.56.204.146,54.353395,18.599154,Gdansk,2026-01-01T00:00:08+00:00,False,NaN,947,15,298.47,0,88787602-704b-4435-99de-caf9d93d3d39,2026-01-01T00:00:08+00:00,1,False,NaN,False,187,3.7.5


In [55]:
df.info()
df.describe(include="all").T.head(30)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 27 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   tx_id                    100000 non-null  object 
 1   sender_id                100000 non-null  object 
 2   recipient_id             100000 non-null  object 
 3   amount                   100000 non-null  float64
 4   currency                 100000 non-null  object 
 5   device_id                100000 non-null  object 
 6   device_type              100000 non-null  object 
 7   device_trusted           100000 non-null  bool   
 8   sender_ip                100000 non-null  object 
 9   sender_lat               100000 non-null  float64
 10  sender_lon               100000 non-null  float64
 11  sender_city              100000 non-null  object 
 12  timestamp                100000 non-null  object 
 13  is_fraud                 100000 non-null  bool   
 14  fraud

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
tx_id,100000,100000,7b01c9a5-b4f6-4721-b6c5-4ed7e07fd8d0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sender_id,100000,9999,eb010eaf-083b-4bd7-a4f5-04d9b7d2e793,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
recipient_id,100000,9999,f58d5bb6-55a2-4675-8e13-c229870183b6,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,100000.0,NaN,NaN,NaN,426.331892,885.327933,0.93,113.7875,237.95,406.865,12972.08
currency,100000,4,USD,25134,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device_id,100000,10705,2a641bd0-24e4-403c-bb79-0f8062eb7b36,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device_type,100000,3,web,33717,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device_trusted,100000,2,True,99294,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sender_ip,100000,99997,156.207.197.196,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sender_lat,100000.0,NaN,NaN,NaN,51.85048,3.218572,6.400553,51.082553,52.213979,52.456974,55.849216


TARGET_CANDIDATES = ["label", "is_fraud", "target", "fraud_label", "fraud_type"]

target_col = next((col for col in TARGET_CANDIDATES if col in df.columns), None)
if target_col is None:
    raise ValueError(
        "No target column found in the training file. Add one column named "
        "'label', 'is_fraud', 'target', 'fraud_label', or 'fraud_type'."
    )

def normalize_target(series: pd.Series) -> pd.Series:
    if series.dtype == "bool":
        return series.astype(int)

    normalized = series.astype(str).str.strip().str.lower()
    false_values = {"", "0", "false", "normal", "none", "nan", "no", "not_fraud"}
    true_values = {"1", "true", "fraud", "yes", "is_fraud"}

    if target_col == "fraud_type":
        return (~normalized.isin(false_values)).astype(int)

    mapped = normalized.map({**{v: 0 for v in false_values}, **{v: 1 for v in true_values}})
    if mapped.isna().any():
        # Named scenario labels such as account_takeover or rapid_fire mean fraud.
        mapped = mapped.fillna((~normalized.isin(false_values)).astype(int))
    return mapped.astype(int)

df["label"] = normalize_target(df[target_col])
target_col = "label"

df[target_col].value_counts(normalize=True).rename("share")


In [56]:
target_col = "label"

if target_col not in df.columns:
    raise ValueError("Expected target column 'label' in the training file.")

if df[target_col].dtype == "bool":
    df[target_col] = df[target_col].astype(int)
elif df[target_col].dtype == "object":
    label_map = {"true": 1, "false": 0, "fraud": 1, "normal": 0, "yes": 1, "no": 0}
    normalized = df[target_col].astype(str).str.strip().str.lower()
    df[target_col] = normalized.map(label_map).fillna(normalized).astype(int)
else:
    df[target_col] = df[target_col].astype(int)

df[target_col].value_counts(normalize=True).rename("share")


label
0    0.9
1    0.1
Name: share, dtype: float64

## 3. Filter And Clean

In [57]:
df_clean = df.copy()

# Basic validity filters.
df_clean = df_clean[df_clean["amount"].notna()]
df_clean = df_clean[df_clean["amount"] > 0]
df_clean = df_clean[df_clean["sender_id"].notna()]
df_clean = df_clean[df_clean["recipient_id"].notna()]
df_clean = df_clean[df_clean["sender_id"] != df_clean["recipient_id"]]

# Parse timestamps.
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], utc=True, errors="coerce")
df_clean["event_timestamp"] = pd.to_datetime(df_clean["event_timestamp"], utc=True, errors="coerce")
df_clean = df_clean[df_clean["timestamp"].notna()]

# Normalize booleans that may arrive from CSV as strings.
for col in ["device_trusted", "device_changed", "is_offhours_login"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})

# Remove exact duplicate transactions if any.
df_clean = df_clean.drop_duplicates(subset=["tx_id"])

print(df.shape, "->", df_clean.shape)
df_clean[target_col].value_counts()

(100000, 27) -> (100000, 27)


label
0    90000
1    10000
Name: count, dtype: int64

## 4. Feature Engineering

In [58]:
df_feat = df_clean.sort_values("timestamp").copy()

df_feat["hour"] = df_feat["timestamp"].dt.hour
df_feat["dayofweek"] = df_feat["timestamp"].dt.dayofweek
df_feat["is_weekend"] = (df_feat["dayofweek"] >= 5).astype(int)
df_feat["amount_log1p"] = np.log1p(df_feat["amount"])
df_feat["amount_to_sender_avg"] = df_feat["amount"] / df_feat["sender_avg_amount"].replace(0, np.nan)
df_feat["event_delay_sec"] = (df_feat["event_timestamp"] - df_feat["timestamp"]).dt.total_seconds()
df_feat["sender_recipient_pair"] = df_feat["sender_id"].astype(str) + "->" + df_feat["recipient_id"].astype(str)

# High-cardinality identifiers and target/leakage columns are dropped for generalization.
drop_cols = [
    "tx_id",
    "event_id",
    "sender_id",
    "recipient_id",
    "device_id",
    "sender_ip",
    "new_device_id",
    "timestamp",
    "event_timestamp",
    "label",
    "is_fraud",
    "fraud_type",
]

X = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])
y = df_feat[target_col].astype(int)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("X:", X.shape)
print("numeric:", numeric_features)
print("categorical:", categorical_features)

X: (100000, 22)
numeric: ['amount', 'device_trusted', 'sender_lat', 'sender_lon', 'sender_account_age_days', 'sender_monthly_tx_count', 'sender_avg_amount', 'pin_failures', 'device_changed', 'is_offhours_login', 'session_duration_sec', 'hour', 'dayofweek', 'is_weekend', 'amount_log1p', 'amount_to_sender_avg', 'event_delay_sec']
categorical: ['currency', 'device_type', 'sender_city', 'app_version', 'sender_recipient_pair']


## 5. Train/Test Split And Preprocessing

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ],
    remainder="drop",
)

## 6. Train Models

In [60]:
models = {
    "logistic_regression": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=5,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

if XGBOOST_AVAILABLE:
    negative = int((y_train == 0).sum())
    positive = int((y_train == 1).sum())
    scale_pos_weight = negative / max(positive, 1)
    models["xgboost"] = Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                XGBClassifier(
                    n_estimators=400,
                    max_depth=5,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    tree_method="hist",
                    scale_pos_weight=scale_pos_weight,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
else:
    print("xgboost is not installed. Install it with: pip install xgboost")

fitted_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model

list(fitted_models)

Training logistic_regression...
Training random_forest...
Training xgboost...


['logistic_regression', 'random_forest', 'xgboost']

## 7. Evaluate Models

In [61]:
def evaluate_model(name, model, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "f1": f1_score(y_test, pred),
        "threshold": threshold,
    }

results = pd.DataFrame(
    [evaluate_model(name, model) for name, model in fitted_models.items()]
).sort_values("roc_auc", ascending=False)

results

,model,roc_auc,f1,threshold
2,xgboost,0.999319,0.934531,0.5
0,logistic_regression,0.997223,0.886249,0.5
1,random_forest,0.992103,0.790660,0.5


In [62]:
best_model_name = results.iloc[0]["model"]
best_model = fitted_models[best_model_name]

proba = best_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Best model:", best_model_name)
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, digits=4))

Best model: xgboost
Confusion matrix:
[[17746   254]
 [   23  1977]]

              precision    recall  f1-score   support

           0     0.9987    0.9859    0.9923     18000
           1     0.8861    0.9885    0.9345      2000

    accuracy                         0.9861     20000
   macro avg     0.9424    0.9872    0.9634     20000
weighted avg     0.9875    0.9861    0.9865     20000



## 8. Optional Threshold Tuning

For fraud detection, the default threshold `0.5` is often not what you want. Tune it against business cost: false positives create review work; false negatives let fraud through.

In [63]:
precision, recall, thresholds = precision_recall_curve(y_test, proba)
threshold_table = pd.DataFrame(
    {
        "threshold": np.r_[thresholds, 1.0],
        "precision": precision,
        "recall": recall,
    }
)
threshold_table["f1"] = 2 * threshold_table["precision"] * threshold_table["recall"] / (
    threshold_table["precision"] + threshold_table["recall"]
).replace(0, np.nan)

threshold_table.sort_values("f1", ascending=False).head(10)

,threshold,precision,recall,f1
18051,0.872720,0.984848,0.9425,0.963209
18050,0.872131,0.984334,0.9425,0.962963
18052,0.873864,0.984841,0.9420,0.962944
18049,0.871543,0.983820,0.9425,0.962717
18053,0.874174,0.984833,0.9415,0.962679
18057,0.875883,0.985849,0.9405,0.962641
18048,0.870276,0.983307,0.9425,0.962471
18054,0.874211,0.984825,0.9410,0.962414
18056,0.875278,0.985333,0.9405,0.962394
18058,0.876212,0.985842,0.9400,0.962375


## 9. Save Best Model